# 05. 생성형 이미지: FLUX.1-dev 직접 실행

RunPod의 NVIDIA RTX 4090(24GB VRAM)에서 **FLUX.1-dev**로 텍스트를 이미지로 생성하는 실습이다.
앞 노트북에 의존하지 않는다. 이미 학습된 모델의 **추론**만 수행하며 가중치를 학습하거나 수정하지 않는다.

- 모델: [black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)
- 이 모델은 **비상업적 사용 라이선스 및 접근 조건**이 있다. Hugging Face에서 접근 승인 후 모델을 RunPod의 `/workspace/models/FLUX.1-dev/`에 내려받는다. 노트북은 로컬 파일만 읽으므로 RunPod에 토큰을 저장하거나 로그인할 필요가 없다.
- 1024×1024 이미지 한 장을 생성한다. 사전에 RunPod에 완전한 Diffusers 형식 모델 디렉터리를 다운로드해야 한다.
- BF16 모델은 무겁다. **4090에서는 sequential CPU offload를 기본값**으로 사용한다. 시스템 RAM 48~64GiB 수준을 권장하며 속도는 느릴 수 있다. 이는 실측 최소 요구 사양이 아니다.

## 준비

RunPod의 CUDA 지원 PyTorch와 Jupyter가 설치돼 있다고 가정한다. 필요한 패키지가 없으면 현재 노트북 커널에서 임시 코드 셀을 만들고 다음을 실행한다. 설치 후 **커널을 재시작**한다.

    %pip install "diffusers==0.35.1" "transformers==4.56.2" "accelerate==1.10.1" "safetensors>=0.4.3" "sentencepiece" "protobuf" Pillow

RunPod의 `/workspace/models/FLUX.1-dev/model_index.json`과 하위 모델 디렉터리가 존재하는지 확인한 다음 아래 셀을 순서대로 실행한다.
패키지 버전은 API 호환성을 위한 설정이다. 실제 RunPod에서 실행을 검증했다는 의미는 아니다.

In [ ]:
import gc
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version

import torch
from diffusers import FluxPipeline
from IPython.display import display

assert torch.cuda.is_available(), "CUDA 지원 PyTorch가 있는 GPU 커널을 선택하세요."
device = torch.device("cuda:0")
torch.cuda.set_device(device)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(device))
free, total = torch.cuda.mem_get_info(device)
print(f"VRAM free/total: {free / 2**30:.2f} / {total / 2**30:.2f} GiB")
print({name: version(name) for name in ["diffusers", "transformers", "accelerate"]})

cache_dir = Path(".cache/huggingface")
output_dir = Path("runs/05_image_generation")
cache_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

## 1. FLUX.1-dev 모델 로딩

모델은 `/workspace/models/FLUX.1-dev/`의 **다운로드 완료된 로컬 파일**을 사용한다.
Hugging Face 계정 인증이나 네트워크 요청 없이 로딩한다. 디렉터리에는 `model_index.json`과 `transformer/`, `text_encoder/`, `text_encoder_2/`, `tokenizer/`, `tokenizer_2/`, `vae/`, `scheduler/` 등이 있어야 한다.
로컬에서 읽는 파일만으로는 원본 커밋 SHA를 확정할 수 없으므로 metrics의 `model_revision`은 null로 기록한다.

4090 24GB에서는 모델 전체를 BF16으로 GPU에 올리면 OOM이 발생할 수 있다.
`offload_mode = "sequential"`이 기본값으로, 하위 모듈을 실행할 때만 GPU로 옮겨 GPU 메모리를 절약한다. **CPU RAM 사용량과 실행 시간이 크게 증가**할 수 있다.
RAM·VRAM 상황에 따라 `"model"`(상대적으로 빠르나 OOM 위험) 또는 `"none"`(모델 전체를 GPU에 올리므로 4090 비권장)을 실험할 수 있다.

모델 로딩 셀을 다시 실행하기 전에 이전 파이프라인 객체와 CUDA 캐시를 정리한다.

In [ ]:
model_id = "/workspace/models/FLUX.1-dev"
revision = None  # 완전히 로컬 파일에서 로드하므로 Hub revision 조회 생략
assert (Path(model_id) / "model_index.json").is_file(), f"모델 파일을 찾을 수 없습니다: {model_id}"
offload_mode = "sequential"  # 4090 24GB 기본값: sequential / model / none
assert offload_mode in {"sequential", "model", "none"}

if "pipe" in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()

pipe = FluxPipeline.from_pretrained(
    model_id,
    revision=revision,
    cache_dir=str(cache_dir),
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    local_files_only=True,
)
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

if offload_mode == "sequential":
    pipe.enable_sequential_cpu_offload()
elif offload_mode == "model":
    pipe.enable_model_cpu_offload()
else:
    pipe = pipe.to(device)

print("Model revision:", revision)
print("Scheduler:", type(pipe.scheduler).__name__)
print("Transformer dtype:", pipe.transformer.dtype)
print("Offload mode:", offload_mode)
print("Loaded allocated MiB:", torch.cuda.memory_allocated(device) / 2**20)

## 2. 생성 설정 바꾸기

- `prompt`: 만들고 싶은 장면. 기본값은 도서관에서 책 읽는 로봇.
- `seed`: 초기 노이즈의 난수 시드. 같은 모델과 설정을 비교할 때 고정한다.
- `steps`: 노이즈 제거 반복 횟수. 모델 학습 epoch가 아니다.
- `guidance_scale`: 프롬프트의 영향을 조절한다. FLUX.1-dev 기본 실습값은 3.5.
- `width`, `height`: 출력 해상도. 기본 1024×1024. 해상도가 높을수록 메모리 및 시간이 늘어난다.
- `max_sequence_length`: 텍스트 인코더가 처리할 프롬프트 최대 길이.

FLUX.1-dev는 CLIP·T5 텍스트 인코더, 확산 Transformer, FlowMatch 계열 스케줄러, VAE를 사용한다.
이 노트북에서는 `negative_prompt`와 SD 1.5 전용 `DDIMScheduler`를 사용하지 않는다.
중간 latent 텐서의 실제 shape는 아래 callback에서 출력한다.

In [ ]:
prompt = "a small robot reading a book in a cozy library, warm light, detailed digital art"
seed = 42
steps = 30
guidance_scale = 3.5
width = 1024
height = 1024
max_sequence_length = 512

assert width > 0 and height > 0 and width % 16 == 0 and height % 16 == 0
assert steps > 0

## 3. 생성 함수: 이미지, VRAM, 실행 시간 및 latent 관찰

매번 CPU generator를 새로 만들어 seed를 초기화한다.
`observe=True`이면 1회차, 매 5회차, 마지막 step에서 callback으로 latent의 shape와 표준편차를 읽는다.
이 값은 이미지 품질 점수가 아니며 일정한 방향으로 변할 필요도 없다.
관찰용 동기화가 추가되므로 **성능 비교에서는 observe=False**로 실행한다.

측정 시간은 모델 다운로드·로딩·PNG 저장을 제외하고 이미지 추론 전체를 포함한다.
VRAM 측정은 PyTorch CUDA allocator 기준이며 CPU offload의 CPU RAM 사용량은 포함하지 않는다.

In [ ]:
def generate(prompt, seed, steps, guidance_scale, width, height, observe=False):
    assert steps > 0 and width > 0 and height > 0 and width % 16 == 0 and height % 16 == 0
    observations = []

    def on_step_end(pipeline, step_index, timestep, callback_kwargs):
        if step_index == 0 or (step_index + 1) % 5 == 0 or step_index + 1 == steps:
            latents = callback_kwargs["latents"]
            row = {
                "step": step_index + 1,
                "timestep": float(timestep.item()) if isinstance(timestep, torch.Tensor) else float(timestep),
                "shape": list(latents.shape),
                "std": latents.float().std().item(),
            }
            observations.append(row)
            print(row)
        return callback_kwargs  # 관찰만 하고 모델의 latent는 변경하지 않음

    kwargs = {}
    if observe:
        kwargs = {
            "callback_on_step_end": on_step_end,
            "callback_on_step_end_tensor_inputs": ["latents"],
        }

    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    with torch.inference_mode():
        result = pipe(
            prompt=prompt,
            generator=torch.Generator(device="cpu").manual_seed(seed),
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            width=width,
            height=height,
            max_sequence_length=max_sequence_length,
            **kwargs,
        )
    torch.cuda.synchronize(device)
    seconds = time.perf_counter() - start

    metrics = {
        "prompt": prompt, "seed": seed, "steps": steps,
        "guidance_scale": guidance_scale, "width": width, "height": height,
        "max_sequence_length": max_sequence_length, "seconds": seconds,
        "allocated_mib": torch.cuda.memory_allocated(device) / 2**20,
        "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20,
        "reserved_mib": torch.cuda.memory_reserved(device) / 2**20,
        "model_id": model_id, "model_revision": revision,
        "scheduler": type(pipe.scheduler).__name__, "offload_mode": offload_mode,
        "gpu": torch.cuda.get_device_name(device), "torch": torch.__version__,
        "cuda": torch.version.cuda, "diffusers": version("diffusers"),
        "transformers": version("transformers"), "accelerate": version("accelerate"),
        "dtype": str(pipe.transformer.dtype), "observations": observations,
    }
    return result.images[0], metrics

## 4. 첫 이미지 생성 및 저장

기본 프롬프트로 1024×1024 이미지 한 장을 생성한다.
이미지를 노트북에 표시하고 PNG 및 실행 설정 JSON을 `runs/05_image_generation/`에 보관한다.
첫 실행은 초기화와 관찰용 callback이 포함되므로 속도 비교의 기준으로 쓰지 않는다.
FLUX 파이프라인은 이 노트북에서 별도의 NSFW 감지 결과 필드를 제공하는 것으로 가정하지 않는다.

In [ ]:
image, metrics = generate(prompt, seed, steps, guidance_scale, width, height, observe=True)
display(image)
print(f"생성 시간: {metrics['seconds']:.2f}s, 최대 할당 VRAM: {metrics['peak_allocated_mib']:.1f} MiB")

run_dir = output_dir / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
image.save(run_dir / "image.png")
(run_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 위치:", run_dir.resolve())

## 5. 선택 실험: 동일 seed에서 20 / 30 / 40 steps 비교

`run_comparison = False`가 기본값이다. `True`로 바꾸면 워밍업 후 세 장을 순차적으로 생성한다.
동일한 prompt·seed·해상도를 유지하고 step 횟수만 바꾸며 속도와 결과물을 비교한다.
Sequential CPU offload는 오래 걸릴 수 있으므로 GPU 대여 시간을 고려해서 실행한다.

In [ ]:
run_comparison = False
if run_comparison:
    warmup_image, _ = generate(prompt, seed, 2, guidance_scale, width, height)
    del warmup_image
    comparison_dir = output_dir / ("compare_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    comparison_dir.mkdir(parents=True, exist_ok=False)
    rows = []
    for step_count in [20, 30, 40]:
        sample, row = generate(prompt, seed, step_count, guidance_scale, width, height)
        display(sample)
        print(f"steps={step_count}, seconds={row['seconds']:.2f}, peak MiB={row['peak_allocated_mib']:.1f}")
        sample.save(comparison_dir / f"steps_{step_count}.png")
        rows.append(row)
    (comparison_dir / "metrics.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print("비교 결과:", comparison_dir.resolve())

## 직접 실험 / 문제 해결

1. seed를 고정하고 프롬프트의 `warm light`를 `moonlight`로 바꿔서 비교한다.
2. seed만 42에서 43으로 바꾼다.
3. `guidance_scale`을 2.5, 3.5, 5.0으로 비교한다.
4. 이미지·시간·VRAM을 함께 보고 step 수와 해상도를 조절한다.

- **401/403**: 모델 페이지에서 조건을 수락했는지와 RunPod의 `hf auth login` 상태를 확인한다.
- **CUDA OOM**: 다른 GPU 커널 종료, `offload_mode="sequential"` 유지, 필요하면 해상도를 768×768로 줄인다. 모드 변경 시 커널 재시작을 권장한다.
- **CPU RAM 부족**: sequential offload가 모델 가중치 대부분을 CPU RAM에 보관한다. RAM 여유를 확인한다.
- **느린 실행**: sequential CPU offload는 레이어 단위 전송이 많다. `offload_mode="model"`로 속도 실험은 가능하지만 4090 24GB에서 OOM이 날 수 있다.
- **import 오류**: 이 노트북의 설치 명령을 현재 Jupyter 커널에서 실행하고 재시작한다.
- **다운로드 실패**: 네트워크 연결, 디스크 용량, 인증을 확인한다.
- `reserved`와 `nvidia-smi`의 사용량은 PyTorch의 현재 할당량과 같지 않을 수 있다.
- 이미지와 측정 JSON은 Git에서 제외되는 `runs/`에 저장된다. 노트북 출력 이미지를 커밋하지 않도록 **Clear All Outputs**를 사용한다.

In [ ]:
release_gpu = False  # 실습이 끝났을 때 True로 변경
if release_gpu:
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    print("파이프라인 해제 완료. 다른 커널이 사용하는 메모리는 별도입니다.")

## 참고 / 검증 범위

- [Diffusers 0.35.1 FluxPipeline API](https://huggingface.co/docs/diffusers/v0.35.1/api/pipelines/flux)
- [FLUX.1-dev 모델 카드와 라이선스](https://huggingface.co/black-forest-labs/FLUX.1-dev)
- [Diffusers 메모리 최적화 / CPU offload](https://huggingface.co/docs/diffusers/v0.35.1/optimization/memory)

이 노트북은 소스의 구조와 API 사용을 기준으로 수정했다. **실제 RunPod RTX 4090에서 모델 로딩·이미지 생성·VRAM 실측은 아직 수행하지 않았다.**
실행 전 Jupyter 커널의 의존성 및 Hugging Face 접근 권한을 확인한다.